# DHO800 Rigol Waveform Examples

**March 2026**

This notebook demonstrates extracting signals from `.wfm` and `.bin` files created by Rigol **DHO800** series oscilloscopes (captured on a DHO824).

The DHO series supports two waveform export formats:

* **`.bin`** - Official binary format documented in the DHO1000 User Guide §19.2.4. Contains calibrated float32 voltage samples for each enabled channel. Identical for DHO800 and DHO1000.
* **`.wfm`** - Proprietary format (reverse-engineered). DHO800 uses a different block encoding from DHO1000, but both are handled transparently by the same parser.

*If RigolWFM is not installed, uncomment the following cell and run (shift-enter)*

In [ ]:
#!pip install --user RigolWFM

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

try:
    import RigolWFM.wfm as rigol
except ModuleNotFoundError:
    print('RigolWFM not installed. To install, uncomment and run the cell above.')

repo = "../wfm/"

## DHO824 - Single-channel `.wfm` capture (CH1)

### Oscilloscope screenshot

Capture taken with CH1 active, measuring a DC signal around 150 mV.

<img src="https://github.com/scottprahl/RigolWFM/raw/main/wfm/DHO824-ch1.png" width="60%">

In [ ]:
filename = "DHO824-ch1.wfm"
w = rigol.Wfm.from_file(repo + filename, 'DHO')

### Textual description

In [ ]:
description = w.describe()
print(description)

### Plot the waveform

In [ ]:
ch = w.channels[0]

plt.plot(ch.times * 1e6, ch.volts * 1000, color='green')
plt.title("%s  CH%d  %.0f Sa/s" % (filename, ch.channel_number, 1 / ch.seconds_per_point))
plt.xlabel("Time (µs)")
plt.ylabel("Voltage (mV)")
plt.grid(True)
plt.show()

## DHO824 - Single-channel `.bin` capture (CH1)

The `.bin` format stores calibrated float32 voltages directly.
No additional scaling is required, and it is identical across all DHO variants.

In [ ]:
filename = "DHO824-ch1.bin"
w = rigol.Wfm.from_file(repo + filename, 'DHO')
print(w.describe())

In [ ]:
ch = w.channels[0]
plt.plot(ch.times * 1e6, ch.volts * 1000, color='green')
plt.title("%s  CH%d" % (filename, ch.channel_number))
plt.xlabel("Time (µs)")
plt.ylabel("Voltage (mV)")
plt.grid(True)
plt.show()

## DHO824 - Two-channel `.bin` capture (CH1 + CH2)

### Oscilloscope screenshot

<img src="https://github.com/scottprahl/RigolWFM/raw/main/wfm/DHO824-ch12.png" width="60%">

In [ ]:
filename = "DHO824-ch12.bin"
w = rigol.Wfm.from_file(repo + filename, 'DHO', selected='12')

colors = ['green', 'red']
active = [ch for ch in w.channels if ch.volts is not None]

fig, axes = plt.subplots(len(active), 1, sharex=True, figsize=(10, 4))
if len(active) == 1:
    axes = [axes]

for ax, ch, color in zip(axes, active, colors):
    ax.plot(ch.times * 1e6, ch.volts * 1000, color=color)
    ax.set_ylabel("mV")
    ax.set_title("CH%d" % ch.channel_number)
    ax.grid(True)

axes[-1].set_xlabel("Time (µs)")
fig.suptitle(filename)
plt.tight_layout()
plt.show()

## DHO824 - Four-channel `.bin` capture (CH1–CH4)

### Oscilloscope screenshot

<img src="https://github.com/scottprahl/RigolWFM/raw/main/wfm/DHO824-ch1234.png" width="60%">

In [ ]:
filename = "DHO824-ch1234.bin"
w = rigol.Wfm.from_file(repo + filename, 'DHO', selected='1234')

colors = ['green', 'red', 'blue', 'orange']
active = [ch for ch in w.channels if ch.volts is not None]
n = len(active)

fig, axes = plt.subplots(n, 1, sharex=True, figsize=(10, 2.5 * n))
if n == 1:
    axes = [axes]

for ax, ch, color in zip(axes, active, colors):
    ax.plot(ch.times * 1e6, ch.volts * 1000, color=color)
    ax.set_ylabel("mV")
    ax.set_title("CH%d  %.2f V/div" % (ch.channel_number, ch.volt_per_division))
    ax.grid(True)

axes[-1].set_xlabel("Time (µs)")
fig.suptitle(filename)
plt.tight_layout()
plt.show()

## DHO824 - Four-channel `.wfm` capture (CH1–CH4)

The `.wfm` parser auto-detects the DHO800 format and deinterleaves the channel data.

In [ ]:
filename = "DHO824-ch1234.wfm"
w = rigol.Wfm.from_file(repo + filename, 'DHO', selected='1234')

colors = ['green', 'red', 'blue', 'orange']
active = [ch for ch in w.channels if ch.volts is not None]
n = len(active)

fig, axes = plt.subplots(n, 1, sharex=True, figsize=(10, 2.5 * n))
if n == 1:
    axes = [axes]

for ax, ch, color in zip(axes, active, colors):
    ax.plot(ch.times * 1e6, ch.volts * 1000, color=color)
    ax.set_ylabel("mV")
    ax.set_title("CH%d  (WFM)" % ch.channel_number)
    ax.grid(True)

axes[-1].set_xlabel("Time (µs)")
fig.suptitle(filename)
plt.tight_layout()
plt.show()

## `.wfm` vs `.bin` comparison for DHO824

Both formats produce identical voltage data (correlation = 1.000, RMS error < 0.006 mV).

In [ ]:
wfm_w = rigol.Wfm.from_file(repo + "DHO824-ch1.wfm", 'DHO', selected='1')
bin_w = rigol.Wfm.from_file(repo + "DHO824-ch1.bin", 'DHO', selected='1')

wfm_v = wfm_w.channels[0].volts
bin_v = bin_w.channels[0].volts
n = min(len(wfm_v), len(bin_v))

corr = np.corrcoef(wfm_v[:n], bin_v[:n])[0, 1]
rms = np.sqrt(np.mean((wfm_v[:n] - bin_v[:n])**2)) * 1000

print(f"CH1 WFM vs BIN:  correlation = {corr:.6f},  RMS error = {rms:.4f} mV")

t = wfm_w.channels[0].times * 1e6
plt.plot(t[:200], wfm_v[:200] * 1000, 'g-', label='WFM', linewidth=2)
plt.plot(t[:200], bin_v[:200] * 1000, 'r--', label='BIN', linewidth=1)
plt.xlabel("Time (µs)")
plt.ylabel("Voltage (mV)")
plt.title("DHO824 CH1: WFM vs BIN (first 200 samples)")
plt.legend()
plt.grid(True)
plt.show()